In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
! pip install datasets pyspark

In [ ]:
import os
import time
import glob
import shutil
import pandas as pd
import numpy as np

OUT_DIR   = "/content/drive/MyDrive/chicago_tableau_exports"
MODEL_DIR = "/content/drive/MyDrive/models"
PRED_DIR  = "/content/drive/MyDrive/preds"
CKPT_DIR  = "/content/drive/MyDrive/spark_checkpoints"

for d in [OUT_DIR, MODEL_DIR, PRED_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

print("Output directories ready.")

Output directories ready.


In [ ]:
# Load dataset from Hugging Face
from datasets import load_dataset

ds = load_dataset("gymprathap/Chicago-Crime-Dataset")
print("Dataset loaded:", ds)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset loaded: DatasetDict({
    train: Dataset({
        features: ['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location'],
        num_rows: 8104658
    })
})


In [ ]:
# Quick pandas EDA on small sample
sample_df = ds["train"].select(range(10000)).to_pandas()
print("Sample shape:", sample_df.shape)
print("Columns:", list(sample_df.columns))
print("\nMissing values (top 10):")
print(sample_df.isnull().sum().sort_values(ascending=False).head(10))
print("\nTop 10 crime types:")
print(sample_df["Primary Type"].value_counts().head(10))
print("\nUnique crime types:", sample_df["Primary Type"].nunique())

Sample shape: (10000, 22)
Columns: ['ID', 'Case Number', 'Date', 'Block', 'IUCR', 'Primary Type', 'Description', 'Location Description', 'Arrest', 'Domestic', 'Beat', 'District', 'Ward', 'Community Area', 'FBI Code', 'X Coordinate', 'Y Coordinate', 'Year', 'Updated On', 'Latitude', 'Longitude', 'Location']

Missing values (top 10):
Longitude               1010
X Coordinate            1010
Location                1010
Y Coordinate            1010
Latitude                1010
Location Description     173
Ward                      13
Community Area            12
Primary Type               0
IUCR                       0
dtype: int64

Top 10 crime types:
Primary Type
THEFT                  2067
BATTERY                1714
CRIMINAL DAMAGE        1131
DECEPTIVE PRACTICE     1037
ASSAULT                 825
MOTOR VEHICLE THEFT     641
OTHER OFFENSE           593
ROBBERY                 401
BURGLARY                341
NARCOTICS               317
Name: count, dtype: int64

Unique crime types: 28

In [ ]:
# Save full dataset as Parquet for Spark
PARQUET_PATH = "chicago_crime.parquet"
ds["train"].to_parquet(PARQUET_PATH)
print("Saved as parquet:", PARQUET_PATH)

Creating parquet from Arrow format:   0%|          | 0/8105 [00:00<?, ?ba/s]

Saved as parquet: chicago_crime.parquet


## PYSPARK DATA ENGINEERING

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Row
from pyspark.sql.functions import broadcast
from pyspark.ml import Transformer

# SparkSession Configuration
# Justification: adaptive execution auto-tunes join strategies and partition

spark = (SparkSession.builder
    .appName("ChicagoCrime_CrimeTypePrediction")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "200")
    .config("spark.executor.memory", "4g")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.cores", "2")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .config("spark.memory.fraction", "0.8")
    .config("spark.memory.storageFraction", "0.3")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setCheckpointDir(CKPT_DIR)

print("Spark version:", spark.version)
print("Spark UI (take screenshot for your report):", sc.uiWebUrl)



Spark version: 4.0.2
Spark UI (take screenshot for your report): http://fe181a128bfc:4040


In [ ]:
# 1B - Load Parquet and inspect schema
df_raw = spark.read.parquet(PARQUET_PATH)
print("Total rows (full dataset):", df_raw.count())
df_raw.printSchema()

Total rows (full dataset): 8104658
root
 |-- ID: long (nullable = true)
 |-- Case Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- IUCR: string (nullable = true)
 |-- Primary Type: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Location Description: string (nullable = true)
 |-- Arrest: boolean (nullable = true)
 |-- Domestic: boolean (nullable = true)
 |-- Beat: long (nullable = true)
 |-- District: long (nullable = true)
 |-- Ward: double (nullable = true)
 |-- Community Area: double (nullable = true)
 |-- FBI Code: string (nullable = true)
 |-- X Coordinate: double (nullable = true)
 |-- Y Coordinate: double (nullable = true)
 |-- Year: long (nullable = true)
 |-- Updated On: string (nullable = true)
 |-- Latitude: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- Location: string (nullable = true)



In [ ]:
df_raw_75 = df_raw.sample(False, 0.75, seed=42)
print("Working dataset rows (75% sample):", df_raw_75.count())

Working dataset rows (75% sample): 6078114


In [ ]:
# Data Validation at Ingestion
cols_to_check = [
    "ID", "Date", "Primary Type",
    "Latitude", "Longitude", "Year", "Beat", "District", "Ward", "Community Area"
]
null_exprs = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols_to_check]
print("Null counts per column (full dataset):")
df_raw.select(null_exprs).show(truncate=False)

total_rows   = df_raw.count()
distinct_ids = df_raw.select("ID").distinct().count()
print("Total rows:", total_rows)
print("Duplicate IDs:", total_rows - distinct_ids)

# All further processing uses the 75% sample only
df_deduped = df_raw_75.dropDuplicates(["ID"])
print("Rows after deduplication (75% sample):", df_deduped.count())

Null counts per column (full dataset):
+---+----+------------+--------+---------+----+----+--------+------+--------------+
|ID |Date|Primary Type|Latitude|Longitude|Year|Beat|District|Ward  |Community Area|
+---+----+------------+--------+---------+----+----+--------+------+--------------+
|0  |0   |0           |89642   |89642    |0   |0   |47      |614831|613455        |
+---+----+------------+--------+---------+----+----+--------+------+--------------+

Total rows: 8104658
Duplicate IDs: 0
Rows after deduplication (75% sample): 6078114


In [ ]:
# Partitioning by Year
PARTITIONED_PATH = "chicago_crime_partitioned_parquet"
(df_deduped
 .repartition("Year")
 .write.mode("overwrite")
 .partitionBy("Year")
 .parquet(PARTITIONED_PATH)
)

# df is the 75% sample, partitioned by Year - used for all ML work
df = spark.read.parquet(PARTITIONED_PATH)
print("Working dataset row count (partitioned, 75%):", df.count())

Working dataset row count (partitioned, 75%): 6078114


In [ ]:
# Distributed Feature Engineering
# Parse date string into timestamp and extract time-based features
df1 = (df
    .withColumn("date_ts",    F.to_timestamp("Date",       "MM/dd/yyyy hh:mm:ss a"))
    .withColumn("updated_ts", F.to_timestamp("Updated On", "MM/dd/yyyy hh:mm:ss a"))
)

# Error handling: drop rows where date did not parse correctly
df1 = df1.filter(F.col("date_ts").isNotNull())

# Extract temporal features - critical predictors for crime type
df1 = (df1
    .withColumn("hour",        F.hour("date_ts"))
    .withColumn("day_of_week", F.dayofweek("date_ts"))
    .withColumn("month",       F.month("date_ts"))
    .withColumn("year_num",    F.col("Year").cast("int"))
)

# Keep only rows with valid coordinates and Primary Type
df2 = df1.filter(
    F.col("Latitude").isNotNull()  &
    F.col("Longitude").isNotNull() &
    F.col("Primary Type").isNotNull()
)
print("Rows with valid data:", df2.count())

Rows with valid data: 6010458


In [ ]:
# 1F - Custom Transformer: GeoGridTransformer
class GeoGridTransformer(Transformer):
    def __init__(self, lat_col="Latitude", lon_col="Longitude",
                 grid_size=0.01, output_col="geo_grid"):
        super().__init__()
        self.lat_col    = lat_col
        self.lon_col    = lon_col
        self.grid_size  = float(grid_size)
        self.output_col = output_col

    def _transform(self, dataset):
        gs  = F.lit(self.grid_size)
        lat = (F.floor(F.col(self.lat_col) / gs) * gs).cast("string")
        lon = (F.floor(F.col(self.lon_col) / gs) * gs).cast("string")
        return dataset.withColumn(self.output_col, F.concat_ws("_", lat, lon))

geo_tf = GeoGridTransformer(grid_size=0.01)
df3    = geo_tf.transform(df2)
print("Sample geo_grid values:")
df3.select("Latitude", "Longitude", "geo_grid").show(5, truncate=False)

Sample geo_grid values:
+------------+-------------+-------------------------------------+
|Latitude    |Longitude    |geo_grid                             |
+------------+-------------+-------------------------------------+
|41.969396938|-87.704349812|41.96_-87.71000000000001             |
|41.7685342  |-87.66546863 |41.76_-87.67                         |
|41.98625526 |-87.658096219|41.980000000000004_-87.66            |
|41.91902511 |-87.700042375|41.910000000000004_-87.71000000000001|
|41.846562186|-87.687034749|41.84_-87.69                         |
+------------+-------------+-------------------------------------+
only showing top 5 rows


In [ ]:
#Broadcast Join: map crime types to broader groups
crime_mapping = [
    Row(primary_type="THEFT",               crime_group="Property"),
    Row(primary_type="BATTERY",             crime_group="Violent"),
    Row(primary_type="CRIMINAL DAMAGE",     crime_group="Property"),
    Row(primary_type="NARCOTICS",           crime_group="Drug"),
    Row(primary_type="ASSAULT",             crime_group="Violent"),
    Row(primary_type="MOTOR VEHICLE THEFT", crime_group="Property"),
    Row(primary_type="ROBBERY",             crime_group="Violent"),
    Row(primary_type="BURGLARY",            crime_group="Property"),
    Row(primary_type="DECEPTIVE PRACTICE",  crime_group="Financial"),
    Row(primary_type="OTHER OFFENSE",       crime_group="Other"),
    Row(primary_type="CRIMINAL TRESPASS",   crime_group="Property"),
    Row(primary_type="PROSTITUTION",        crime_group="Other"),
    Row(primary_type="WEAPONS VIOLATION",   crime_group="Violent"),
    Row(primary_type="PUBLIC PEACE VIOLATION", crime_group="Other"),
    Row(primary_type="CRIM SEXUAL ASSAULT", crime_group="Violent"),
    Row(primary_type="ARSON",               crime_group="Property"),
]
map_df = spark.createDataFrame(crime_mapping)

df3 = (df3
    .join(broadcast(map_df), df3["Primary Type"] == map_df["primary_type"], "left")
    .drop("primary_type")
    .fillna("Other", subset=["crime_group"])
)
print("Crime group distribution:")
df3.groupBy("crime_group").count().orderBy("count", ascending=False).show()

Crime group distribution:
+-----------+-------+
|crime_group|  count|
+-----------+-------+
|   Property|2758166|
|    Violent|1838581|
|      Other| 597540|
|       Drug| 556079|
|  Financial| 260092|
+-----------+-------+



In [ ]:
# Dashboard 1 Exports: Data Quality

# Missing values per column (use full dataset for accurate reporting)
all_cols      = df_raw.columns
null_ex       = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in all_cols]
null_vals     = df_raw.select(null_ex).collect()[0].asDict()
total         = df_raw.count()
missing_rows  = [
    (c, int(null_vals[c]), round(float(null_vals[c]) / total * 100, 2))
    for c in all_cols
]
missing_df = spark.createDataFrame(missing_rows, ["column_name", "null_count", "null_percent"])
(missing_df.orderBy(F.desc("null_percent"))
 .coalesce(1).write.mode("overwrite").option("header", True)
 .csv(OUT_DIR + "/dash1_missing_values"))
print("Saved: dash1_missing_values")

# Row count by year (use full dataset for accurate reporting)
(df_raw.groupBy("Year").count().orderBy("Year")
 .coalesce(1).write.mode("overwrite").option("header", True)
 .csv(OUT_DIR + "/dash1_rows_by_year"))
print("Saved: dash1_rows_by_year")

# Date parse quality (use full dataset for accurate reporting)
df_dates = df_raw.withColumn("date_ts", F.to_timestamp("Date", "MM/dd/yyyy hh:mm:ss a"))
(df_dates.groupBy("Year")
 .agg(
     F.count("*").alias("total_rows"),
     F.sum(F.col("date_ts").isNotNull().cast("int")).alias("parsed_ok"),
     F.sum(F.col("date_ts").isNull().cast("int")).alias("parsed_fail")
 )
 .orderBy("Year")
 .coalesce(1).write.mode("overwrite").option("header", True)
 .csv(OUT_DIR + "/dash1_date_parse_quality"))
print("Saved: dash1_date_parse_quality")

# Top crime types distribution (use full dataset for accurate reporting)
(df_raw.groupBy("Primary Type").count()
 .withColumnRenamed("count", "crime_count")
 .orderBy(F.desc("crime_count"))
 .coalesce(1).write.mode("overwrite").option("header", True)
 .csv(OUT_DIR + "/dash1_crime_type_distribution"))
print("Saved: dash1_crime_type_distribution")

# Crime count by year (use full dataset for accurate reporting)
(df_raw.groupBy("Year", "Primary Type").count()
 .withColumnRenamed("count", "crime_count")
 .orderBy("Year", F.desc("crime_count"))
 .coalesce(1).write.mode("overwrite").option("header", True)
 .csv(OUT_DIR + "/dash1_crime_by_year_and_type"))
print("Saved: dash1_crime_by_year_and_type")


Saved: dash1_missing_values
Saved: dash1_rows_by_year
Saved: dash1_date_parse_quality
Saved: dash1_crime_type_distribution
Saved: dash1_crime_by_year_and_type
